# P4 — Evasión de Colisiones (MyCobot 280)

**Autor:** Alex — Ingeniero de Control y Colisiones
**Curso:** ROB 2026

## Contenido
1. Conexión al robot
2. Poses clave + variable de emergencia
3. Definición de límites (Z mínimo y rangos articulares)
4. Funciones de verificación y movimiento seguro
5. Pruebas de evasión (escenario mesa y escenario ángulos)
6. Sistema avanzado de límites en coordenadas cartesianas
7. Pruebas de límites cartesianos


In [ ]:
from pymycobot.mycobot import MyCobot
import time

mc = MyCobot('/dev/ttyUSB0', 1000000)
mc.power_on()
time.sleep(1)

print('Angles:', mc.get_angles())
print('Coords:', mc.get_coords())
print('Robot conectado')

In [ ]:
# ============================================================
# POSES CLAVE + POSE INICIAL
# ============================================================

# Poses del ciclo principal
pose_inicial   = [0.7,    -0.17,  -0.61,  -1.58,  -0.35, -44.29]
pose_observar  = [-53.87, -56.95, -7.38,  -31.28,  3.33, -27.86]
pose_agarrar   = [-52.55, -71.01, -7.38,  -31.11,  3.25, -27.77]
pose_alto      = [-52.55, -50.09, -6.85,  -31.11,  3.33, -27.94]
pose_mover     = [99.05,  -33.13, -48.51,  -7.47,  3.77, -44.82]
pose_depositar = [99.4,   -48.33, -48.6,   -6.94,  3.33, -44.38]

# Variable de parada de emergencia
PARAR = False

# Mover a pose inicial
mc.send_angles(pose_inicial, 30)
time.sleep(3)

print("Poses cargadas correctamente")
print("pose_inicial   :", pose_inicial)
print("pose_observar  :", pose_observar)
print("pose_agarrar   :", pose_agarrar)
print("pose_alto      :", pose_alto)
print("pose_mover     :", pose_mover)
print("pose_depositar :", pose_depositar)
print("Angles:", mc.get_angles())
print("Coords:", mc.get_coords())

In [ ]:
angulos = mc.get_angles()
coords  = mc.get_coords()

print(f"ANGULOS: {angulos}")
print(f"COORDS:  {coords}")

## Sistema de Evasion de Colisiones

**Escenario 1 — Colisión con la mesa:**
El robot fue movido manualmente hasta tocar la mesa, donde se midió Z = 116 mm.
Se define un margen de seguridad: `Z_MIN = 130 mm`.

**Escenario 2 — Límites articulares:**
Se aplican los rangos físicos de cada articulación del MyCobot 280.


In [ ]:
# ============================================================
# P4: EVASION DE COLISIONES
# Escenario 1 — Colision con la mesa (Z minimo)
# Escenario 2 — Limites articulares J1-J6
# ============================================================

Z_MIN = 130  # mm — robot choca en Z=116mm, margen seguro

LIMITES = {
    'J1': (-168, 168),
    'J2': (-135, 90),
    'J3': (-150, 150),
    'J4': (-145, 145),
    'J5': (-165, 165),
    'J6': (-180, 180),
}

def verificar_colision_mesa():
    coords = mc.get_coords()
    if coords is None:
        print("No se pudo leer coordenadas")
        return False
    z_actual = coords[2]
    if z_actual < Z_MIN:
        print(f"COLISION CON MESA DETECTADA")
        print(f"   Z actual:  {z_actual}mm")
        print(f"   Z minimo:  {Z_MIN}mm")
        print(f"   Volviendo a pose segura...")
        mc.send_angles(pose_inicial, 20)
        time.sleep(3)
        return False
    print(f"Altura segura: Z={z_actual}mm (min={Z_MIN}mm)")
    return True

def verificar_colision_angulos(pose):
    for i, angulo in enumerate(pose):
        lo, hi = LIMITES[f'J{i+1}']
        if not (lo <= angulo <= hi):
            print(f"ANGULO FUERA DE LIMITE DETECTADO")
            print(f"   J{i+1}: {angulo} grados — limite: [{lo}, {hi}]")
            print(f"   Movimiento bloqueado")
            return False
    print(f"Angulos seguros: {pose}")
    return True

def mover_seguro(pose, nombre="pose", velocidad=20):
    global PARAR
    if PARAR:
        print("PARADA DE EMERGENCIA activa")
        return False
    if not verificar_colision_angulos(pose):
        return False
    print(f"Moviendo a {nombre}...")
    mc.send_angles(pose, velocidad)
    time.sleep(3)
    if not verificar_colision_mesa():
        return False
    print(f"{nombre}: {mc.get_angles()}")
    return True

print("P4 cargado")
print(f"   Escenario 1: Mesa — choca en Z=116mm, limite seguro={Z_MIN}mm")
print(f"   Escenario 2: Limites articulares J1-J6 del MyCobot 280")

## Pruebas de Evasion

Tres pruebas para validar el sistema:
1. Ángulo fuera de límite (debe bloquearse).
2. Bajar hasta el límite seguro de Z (debe detectarse).
3. Movimiento seguro normal (debe completarse).


In [ ]:
# ============================================================
# PRUEBAS DE EVASION DE COLISIONES
# ============================================================

# Posicion limite medida fisicamente (Z=124.7mm, justo antes de chocar)
pose_limite_seguro = [-3.77, -77.51, -1.75, -10.63, 6.76, -43.59]

# PRUEBA 1: angulo fuera de limite — debe bloquearse
print("=" * 50)
print("PRUEBA 1: Angulo invalido (debe bloquearse)")
print("=" * 50)
pose_invalida = [0, -200, 0, 0, 0, 0]
verificar_colision_angulos(pose_invalida)

# PRUEBA 2: bajar hasta el limite seguro
print("\n" + "=" * 50)
print("PRUEBA 2: Limite de mesa (debe detectar colision)")
print("=" * 50)
print(f"Moviendo a Z=124.7mm (choca en Z=116mm, limite={Z_MIN}mm)...")
mover_seguro(pose_limite_seguro, "LIMITE SEGURO")

# PRUEBA 3: movimiento seguro normal
print("\n" + "=" * 50)
print("PRUEBA 3: Movimiento seguro (debe completarse)")
print("=" * 50)
mover_seguro(pose_observar, "OBSERVAR")

# Volver a pose inicial
mover_seguro(pose_inicial, "INICIAL")

## Limites Cartesianos (Avanzado)

Además de los ángulos, se verifican los límites de las coordenadas cartesianas (X, Y, Z, RX, RY, RZ) medidas físicamente en el laboratorio.


In [ ]:
# ============================================================
# LIMITES FISICOS MEDIDOS EN LABORATORIO
# ============================================================

LIMITE_COORDS = {
    'X': (-280, 280),
    'Y': (-280, 280),
    'Z': (125,  419),
    'RX': (-180, 180),
    'RY': (-180, 180),
    'RZ': (-180, 180),
}

def verificar_coords_seguras(coords):
    """Verifica que las coordenadas esten dentro de los limites seguros"""
    ejes = ['X', 'Y', 'Z', 'RX', 'RY', 'RZ']
    for i, eje in enumerate(ejes):
        lo, hi = LIMITE_COORDS[eje]
        valor = coords[i]
        if not (lo <= valor <= hi):
            print(f"ALERTA DE EMERGENCIA")
            print(f"   Eje {eje}: {valor} fuera de limite [{lo}, {hi}]")
            print(f"   Deteniendo robot y volviendo a pose segura...")
            mc.send_angles(pose_inicial, 20)
            time.sleep(3)
            print(f"   Robot en pose segura")
            return False
    return True

def mover_con_limites(pose, nombre="pose", velocidad=20):
    """Mueve el robot y verifica limites en todos los ejes"""
    global PARAR
    if PARAR:
        print("PARADA DE EMERGENCIA activa")
        return False

    if not verificar_colision_angulos(pose):
        return False

    print(f"Moviendo a {nombre}...")
    mc.send_angles(pose, velocidad)
    time.sleep(3)

    coords = mc.get_coords()
    if coords is None:
        print("No se pudo leer coordenadas")
        return False

    print(f"Coords actuales: {coords}")

    if not verificar_coords_seguras(coords):
        return False

    print(f"{nombre} alcanzada dentro de limites seguros")
    return True

print("Control de limites cargado")
print(f"   Z  -> [{LIMITE_COORDS['Z'][0]}mm, {LIMITE_COORDS['Z'][1]}mm]")
print(f"   X  -> [{LIMITE_COORDS['X'][0]}mm, {LIMITE_COORDS['X'][1]}mm]")
print(f"   Y  -> [{LIMITE_COORDS['Y'][0]}mm, {LIMITE_COORDS['Y'][1]}mm]")

In [ ]:
# ============================================================
# PRUEBAS DE LIMITES DE COORDENADAS
# ============================================================

# PRUEBA 1: pose normal — debe pasar
print("=" * 50)
print("PRUEBA 1: Pose observar (segura)")
print("=" * 50)
mover_con_limites(pose_observar, "OBSERVAR")

# PRUEBA 2: bajar hasta limite seguro Z
print("\n" + "=" * 50)
print("PRUEBA 2: Limite minimo Z")
print("=" * 50)
pose_limite_seguro = [-3.77, -77.51, -1.75, -10.63, 6.76, -43.59]
mover_con_limites(pose_limite_seguro, "LIMITE Z")

# PRUEBA 3: angulo invalido — debe bloquearse
print("\n" + "=" * 50)
print("PRUEBA 3: Angulo invalido")
print("=" * 50)
pose_invalida = [0, -200, 0, 0, 0, 0]
mover_con_limites(pose_invalida, "INVALIDA")

# PRUEBA 4: volver a inicial
print("\n" + "=" * 50)
print("PRUEBA 4: Volver a pose inicial")
print("=" * 50)
mover_con_limites(pose_inicial, "INICIAL")